In [1]:
using Serialization, Statistics, Distributions, StableRNGs, CSV, ForwardDiff, GlobalSensitivity
include("misc.jl")
include("ode_problem.jl")
include("target_probability.jl")
include("marginal_kl_divergence.jl")
include("define_likelihood_data.jl")
include("sensitivity_analysis.jl");

RNG

In [3]:
rng = StableRNG(34627);

Input Run Details

In [ ]:
#for analysis needing pairs (augmented and baseline comparison)
#augmented_run = "220"
#baseline_run = "201"
#prior_nonbinding_offset = [2,2]
#prior_binding_offset = [0,0];
#prior_bounds = return_prior_bounds_empirical(prior_nonbinding_offset, prior_binding_offset);

In [35]:
#for analysis needing pairs (augmented and baseline comparison)
#augmented_run = "204"
#baseline_run = "205"
#prior_nonbinding_offset  = [1,1]
#prior_binding_offset = [0,0] #will be set later
#bound_offset = [1,1]
#binding_bound_offset = [0,0]; #will be set later
#prior_bounds = return_prior_bounds_empirical(bound_offset, binding_bound_offset) #for target probability
#redefine narrow koff bounds
#koff_i = return_koff_indices()
#prior_bounds[koff_i] = return_prior_bounds_empirical(bound_offset, [-2,0])[koff_i] 
#redefine k12b bounds, so that reported value is within the prior bounds
#prior_bounds[koff_i[2]] = return_prior_bounds_empirical(bound_offset, [-1,-1])[koff_i[2]] 
#redefine narrow kon bounds
#kon_i = return_kon_indices()
#prior_bounds[kon_i] = return_prior_bounds_empirical(bound_offset, [-1,-1])[kon_i]
#redefine k13f bounds, so that reported value is within the prior bounds
#prior_bounds[kon_i[3]] = return_prior_bounds_empirical(bound_offset, [-2,0])[kon_i[3]];

In [36]:
#augmented_run = "202"
#baseline_run = "203"
#bound_offset = [3,3]
#binding_bound_offset = [1,1]
#prior_bounds = return_prior_bounds_empirical(bound_offset, binding_bound_offset); #for target probability

50-element Vector{Vector{Float64}}:
 [-0.4685210829577451, 5.531478917042255]
 [-1.3010299956639813, 4.698970004336019]
 [-1.0, 5.0]
 [-2.769551078621726, 3.230448921378274]
 [-0.3467874862246565, 5.653212513775344]
 [-3.0, 3.0]
 [-7.0, 1.0]
 [-5.0, 1.0]
 [-5.346787486224656, 0.6532125137753435]
 [-4.522878745280337, 1.4771212547196624]
 [-7.0, 1.0]
 [-5.0, 1.0]
 [-7.0, 1.0]
 ⋮
 [-4.0, 2.0]
 [-5.0, 1.0]
 [-5.0, 1.0]
 [-3.0, 3.0]
 [-7.0, 1.0]
 [-5.0, 1.0]
 [-4.301029995663981, 1.6989700043360187]
 [-3.0, 3.0]
 [-5.221848749616356, 0.7781512503836439]
 [-3.5228787452803374, 2.4771212547196626]
 [-7.0, 1.0]
 [-5.0, 1.0]

Define Prior for Analyses

Calculate Quantiles of Posteriors

In [37]:
#load posterior samples - change name as necessary 
aug_samples = deserialize("outputs/300_$(augmented_run)_posterior_samples_thinned.jls") #ndims x nsamples
baseline_samples = deserialize("outputs/300_$(baseline_run)_posterior_samples_thinned.jls") #ndims x nsamples 
n_parameters, n_samples = size(aug_samples)

#define priors
#prior_bounds = return_prior_bounds_empirical(prior_nonbinding_offset, prior_binding_offset)

#parameter names
parameter_names = return_inferred_parameters()

#ground truth value on log10 scale
ground_truth = log10.(groundtruth_parameter_values())

#calculate quantiles
lower_bound = 0.05
upper_bound = 0.95
my_quantiles = [[Statistics.quantile(baseline_samples[i,:], lower_bound), Statistics.quantile(baseline_samples[i,:], upper_bound)] for i in 1:n_parameters]
my_quantiles_aug = [[Statistics.quantile(aug_samples[i,:], lower_bound), Statistics.quantile(aug_samples[i,:], upper_bound)] for i in 1:n_parameters]
my_quantiles_priors = []
for i in 1:n_parameters
    bounds = prior_bounds[i]
    difference = bounds[2] - bounds[1]
    quantiles_offset = difference*lower_bound
    push!(my_quantiles_priors, [bounds[1]+quantiles_offset, bounds[2]-quantiles_offset])
end

#save 
serialize("outputs/600_$(baseline_run)_posterior_quantiles.jls", my_quantiles)
serialize("outputs/600_$(augmented_run)_posterior_quantiles.jls", my_quantiles_aug)
serialize("outputs/600_$(augmented_run)_prior_quantiles.jls", my_quantiles_priors)

Mean Absolute Error w.r.t. Posterior Samples

In [38]:
#load posterior samples
aug_samples = deserialize("outputs/300_$(augmented_run)_posterior_samples_thinned.jls") #ndims x nsamples
baseline_samples = deserialize("outputs/300_$(baseline_run)_posterior_samples_thinned.jls") #ndims x nsamples 
n_parameters, n_samples = size(aug_samples)

#parameter names
parameter_names = return_inferred_parameters()

#ground truth value on log10 scale
ground_truth = log10.(groundtruth_parameter_values())

#calculate absolute error of each sample
abs_error = Base.stack([abs.(baseline_samples[:,i] .- ground_truth) for i in 1:n_samples])
abs_error_reg = Base.stack([abs.(aug_samples[:,i] .- ground_truth) for i in 1:n_samples])

#take mean of absolute errors for each parameter
mae = [sum(abs_error[i,:])/n_samples for i in 1:n_parameters]
mae_reg = [sum(abs_error_reg[i,:])/n_samples for i in 1:n_parameters]

#convert to dictionary
mae_dict = Dict(parameter_names[i] => mae[i] for i in 1:n_parameters)
mae_aug_dict = Dict(parameter_names[i] => mae_reg[i] for i in 1:n_parameters)

#save 
serialize("outputs/600_$(baseline_run)_mean_absolute_error.jls", mae_dict)
serialize("outputs/600_$(augmented_run)_mean_absolute_error.jls", mae_aug_dict)

Calculate KL Divergence for From Augmented to Baseline Posterior

In [39]:
#load posterior samples
aug_samples = deserialize("outputs/300_$(augmented_run)_posterior_samples_thinned.jls") #ndims x nsamples
baseline_samples = deserialize("outputs/300_$(baseline_run)_posterior_samples_thinned.jls") #ndims x nsamples 
n_parameters, n_samples = size(aug_samples)
#prior_bounds = return_prior_bounds_empirical(prior_nonbinding_offset, prior_binding_offset)
prior_distributions = [Uniform(prior_bounds[i]...) for i in 1:n_parameters]
prior_samples = transpose(Base.stack([rand(rng, prior_distributions[i], n_samples) for i in 1:n_parameters])) #ndims x nsamples 

#parameter names
parameter_names = return_inferred_parameters()
kl_div_augmented_baseline, parameter_names = kl_divergence_bits_samples(aug_samples, baseline_samples, parameter_names)
kl_div_baseline_prior, parameter_names = kl_divergence_bits_samples(baseline_samples, prior_samples, parameter_names)

#save
serialize("outputs/600_$(augmented_run)_$(baseline_run)_kl_div.jls", kl_div_augmented_baseline)
serialize("outputs/600_$(baseline_run)_prior_kl_div.jls", kl_div_augmented_baseline)

Calculate quantiles of finegrain predictions for which we have experimental data

In [40]:
data_files = readdir("data")
sort_files = data_files .!= ".DS_Store"
data_files = sort(data_files[sort_files]) #sort to ensure consistent order
sort_files = data_files .!= "kholodenko1.xml"
data_files = sort(data_files[sort_files])
sort_files = data_files .!= "predicted_kd.csv"
data_files = sort(data_files[sort_files])
quantity_names = [replace(data_files[i], ".csv" => "") for i in 1:length(data_files)];

In [41]:
for r in [augmented_run, baseline_run]    
    for i in quantity_names
        predictions = Base.stack(deserialize("outputs/500_$(r)_finegrain_predictions_$(i).jls")) #ntime x n
        n_time_points, n_predictions = size(predictions)
        uq = 0.95
        lq = 0.05
        m = 0.5
        quantile_up = [Statistics.quantile(predictions[i,:], uq) for i in 1:n_time_points]
        quantile_low = [Statistics.quantile(predictions[i,:], lq) for i in 1:n_time_points]
        median = [Statistics.quantile(predictions[i,:], m) for i in 1:n_time_points]
        serialize("outputs/600_$(r)_prediction_quantiles_$(i).jls", Dict("lower_quantile"=>quantile_low, 
        "median"=>median, "upper_quantile" =>quantile_up))
    end
end

Calculate Quantiles of Predictions for All Species

In [42]:
#calculate quantiles
for r in [augmented_run, baseline_run]
    all_predictions = deserialize("outputs/500_$(r)_predictions.jls")
    n_doses, n_samples = size(all_predictions)
    n_species = length(all_predictions[1].u[1])
    n_time_points = length(all_predictions[1].u);
    reshaped_predictions = []
    for i in 1:n_doses
        p_per_dose = all_predictions[i,:]
        for z in 1:n_species
            species_dose = zeros(n_time_points, n_samples)
            for j in 1:n_samples
                species_dose[:,j] = Base.stack(p_per_dose[j].u)[z,:]
            end
            reshaped_predictions = cat(reshaped_predictions, [Base.stack(species_dose)], dims=1)
        end
    end
    quantiles = []   
    for j in 1:(n_doses*n_species)
        predictions = reshaped_predictions[j]
        uq = 0.95
        lq = 0.05
        m = 0.5
        quantile_up = [Statistics.quantile(predictions[i,:], uq) for i in 1:n_time_points]
        quantile_low = [Statistics.quantile(predictions[i,:], lq) for i in 1:n_time_points]
        median = [Statistics.quantile(predictions[i,:], m) for i in 1:n_time_points]
        quantiles = cat(quantiles, [Dict("lower_quantile"=>quantile_low, "median"=>median, "upper_quantile" =>quantile_up)], dims=1)
    end
    serialize("outputs/600_$(r)_prediction_quantiles.jls", quantiles)
end

In [43]:
#calculate quantiles for finegrain predictions - this is for plotting
for r in [augmented_run, baseline_run]
    all_predictions = deserialize("outputs/500_$(r)_predictions_finegrain.jls")
    n_doses, n_samples = size(all_predictions)
    n_species = length(all_predictions[1].u[1])
    n_time_points = length(all_predictions[1].u);
    reshaped_predictions = []
    for i in 1:n_doses
        p_per_dose = all_predictions[i,:]
        for z in 1:n_species
            species_dose = zeros(n_time_points, n_samples)
            for j in 1:n_samples
                species_dose[:,j] = Base.stack(p_per_dose[j].u)[z,:]
            end
            reshaped_predictions = cat(reshaped_predictions, [Base.stack(species_dose)], dims=1)
        end
    end
    quantiles = []   
    for j in 1:(n_doses*n_species)
        predictions = reshaped_predictions[j]
        uq = 0.95
        lq = 0.05
        m = 0.5
        quantile_up = [Statistics.quantile(predictions[i,:], uq) for i in 1:n_time_points]
        quantile_low = [Statistics.quantile(predictions[i,:], lq) for i in 1:n_time_points]
        median = [Statistics.quantile(predictions[i,:], m) for i in 1:n_time_points]
        quantiles = cat(quantiles, [Dict("lower_quantile"=>quantile_low, "median"=>median, "upper_quantile" =>quantile_up)], dims=1)
    end
    serialize("outputs/600_$(r)_prediction_quantiles_finegrain.jls", quantiles)
end 

Local Sensitivity Analysis

In [12]:
#first, extract parameter set that maximizes baseline training data likelihood
likelihood = deserialize("outputs/300_$(baseline_run)_posterior_samples_thinned_likelihood.jls");
max_likelihood = maximum(likelihood)
mask = likelihood .== max_likelihood
max_likelihood_parameters = deserialize("outputs/300_$(baseline_run)_posterior_samples_thinned.jls")[:, mask]

#calculate local sensitivity of each species about this parameter value 
egf_doses = return_ligand_dose_order_for_likelihood_w_pshc_test()
n_doses = length(egf_doses)
n_species = 23
solver_inputs = return_ode_problem_solver_default_inputs()
solver_inputs["reltol"] = 1.0e-12
solver_inputs["abstol"] = 1.0e-9
local_sensitivity_kb = []
local_sensitivity_kf = []
local_sensitivity_nonbinding = []
for i in 1:n_doses 
    odesys, u0, tspan, p = return_ode_problem_default_inputs(egf_doses[i])
    odeprob = DifferentialEquations.ODEProblem(odesys, u0, tspan, p) #note, parameters will be redefined in next function
    for j in 1:n_species
        global local_sensitivity_kb
        global local_sensitivity_kf
        global local_sensitivity_nonbinding
        local_sensitivity = ForwardDiff.jacobian(p -> local_sensitivity_analysis(p, odeprob, solver_inputs, j), max_likelihood_parameters) #take parameters out of log10scale first
        local_sensitivity_kb = cat(local_sensitivity_kb, [local_sensitivity[:,return_koff_indices()]], dims=1)
        local_sensitivity_kf = cat(local_sensitivity_kf, [local_sensitivity[:, return_kon_indices()]], dims=1)
        local_sensitivity_nonbinding = cat(local_sensitivity_nonbinding, [local_sensitivity[:, return_nonbinding_indices()]], dims=1)
    end
end
serialize("outputs/600_$(baseline_run)_local_sensitivity_kb.jls", local_sensitivity_kb)
serialize("outputs/600_$(baseline_run)_local_sensitivity_kf.jls", local_sensitivity_kf)
serialize("outputs/600_$(baseline_run)_local_sensitivity_nonbind.jls", local_sensitivity_nonbinding)


#reshape and save for correlation calculation between change in median and local sensitivity of parameters
#calculate absolute difference between medians for each output of interest
augmented_quantiles = deserialize("outputs/600_$(augmented_run)_prediction_quantiles.jls")[1:end] # [1:46] exclude third dose. Third dose has only one training species timecourse, EGFR, and that timecourse is not well recapitulated
baseline_quantiles = deserialize("outputs/600_$(baseline_run)_prediction_quantiles.jls")[1:end] #exclude third dose. Third dose has only one training species timecourse, EGFR, and that timecourse is not well recapitulated
n_outputs = length(augmented_quantiles) #n_species x n_doses
n_timepoints = length(augmented_quantiles[1]["median"])
difference_between_medians = []
for i in 1:n_outputs
    global difference_between_medians
    difference_between_medians = cat(difference_between_medians, [abs.(augmented_quantiles[i]["median"] .- baseline_quantiles[i]["median"])], dims=1)
end
#save
med_dif = reshape(Base.stack(difference_between_medians), n_outputs*n_timepoints)
serialize("outputs/600_$(baseline_run)_$(augmented_run)_difference_between_medians_1d.jls", med_dif)

#load local sensitivity to parameter for each output of interest, reshape to match median difference, save
for i in ["kb", "kf", "nonbind"]
    local_sensitivity_kb = deserialize("outputs/600_$(baseline_run)_local_sensitivity_$(i).jls") #vector of n_outputs, each output is sensitivity matrix of n_timepoints x n_parameters
    #maximum across kb
    max_ls_kb = []
    for i in 1:n_outputs
        max_ls_kb = cat(max_ls_kb, [vec(maximum(abs.(local_sensitivity_kb[i]), dims=2))], dims=1) #take parameter with highest sensitivity for particular output
    end
    ls_kb = reshape(Base.stack(max_ls_kb), n_outputs*n_timepoints)
    serialize("outputs/600_$(baseline_run)_local_sensitivity_$(i)_1d.jls", ls_kb)
end

Relative Local Sensitivity Analysis

In [41]:
#first, extract parameter set that maximizes baseline training data likelihood
likelihood = deserialize("outputs/300_$(baseline_run)_posterior_samples_thinned_likelihood.jls");
max_likelihood = maximum(likelihood)
mask = likelihood .== max_likelihood
max_likelihood_parameters = deserialize("outputs/300_$(baseline_run)_posterior_samples_thinned.jls")[:, mask]

#calculate local sensitivity of each species about this parameter value 
egf_doses = return_ligand_dose_order_for_likelihood_w_pshc_test()
n_doses = length(egf_doses)
n_species = 23
solver_inputs = return_ode_problem_solver_default_inputs()
solver_inputs["reltol"] = 1.0e-12
solver_inputs["abstol"] = 1.0e-9
local_sensitivity_kb = []
local_sensitivity_kf = []
local_sensitivity_nonbinding = [] 
for i in 1:n_doses 
    odesys, u0, tspan, p = return_ode_problem_default_inputs(egf_doses[i])
    odeprob = DifferentialEquations.ODEProblem(odesys, u0, tspan, p) #note, parameters will be redefined in next function
    for j in 1:n_species
        local_sensitivity = ForwardDiff.jacobian(p -> local_sensitivity_analysis_absolute(p, odeprob, solver_inputs, j), 10.0.^max_likelihood_parameters) #take parameters out of log10scale first
        #multiple by 5% change in parameter value 
        n_outputs, n_parameters = size(local_sensitivity)
        percent_change = transpose(reshape(repeat((10.0.^max_likelihood_parameters).*0.05, n_outputs),n_parameters, n_outputs))
        local_sensitivity = local_sensitivity.*percent_change
        global local_sensitivity_kb
        global local_sensitivity_kf
        global local_sensitivity_nonbinding
        local_sensitivity_kb = cat(local_sensitivity_kb, [local_sensitivity[:,return_koff_indices()]], dims=1)
        local_sensitivity_kf = cat(local_sensitivity_kf, [local_sensitivity[:, return_kon_indices()]], dims=1)
        local_sensitivity_nonbinding = cat(local_sensitivity_nonbinding, [local_sensitivity[:, return_nonbinding_indices()]], dims=1)
    end
end
serialize("outputs/600_$(baseline_run)_5_percent_change_kb.jls", local_sensitivity_kb)
serialize("outputs/600_$(baseline_run)_5_percent_change_kf.jls", local_sensitivity_kf)
serialize("outputs/600_$(baseline_run)_5_percent_change_nonbind.jls", local_sensitivity_nonbinding)


#reshape and save for correlation calculation between change in median and local sensitivity of parameters
#calculate absolute difference between medians for each output of interest
augmented_quantiles = deserialize("outputs/600_$(augmented_run)_prediction_quantiles.jls")[1:end] # [1:46] exclude third dose. Third dose has only one training species timecourse, EGFR, and that timecourse is not well recapitulated
baseline_quantiles = deserialize("outputs/600_$(baseline_run)_prediction_quantiles.jls")[1:end] #exclude third dose. Third dose has only one training species timecourse, EGFR, and that timecourse is not well recapitulated
n_outputs = length(augmented_quantiles) #n_species x n_doses
n_timepoints = length(augmented_quantiles[1]["median"])
difference_between_medians = []
for i in 1:n_outputs
    global difference_between_medians
    difference_between_medians = cat(difference_between_medians, [abs.(augmented_quantiles[i]["median"] .- baseline_quantiles[i]["median"])], dims=1)
end
#save
med_dif = reshape(Base.stack(difference_between_medians), n_outputs*n_timepoints)
serialize("outputs/600_$(baseline_run)_$(augmented_run)_difference_between_medians_1d.jls", med_dif)

#load local sensitivity to parameter for each output of interest, reshape to match median difference, save
for i in ["kb", "kf", "nonbind"]
    local_sensitivity_kb = deserialize("outputs/600_$(baseline_run)_5_percent_change_$(i).jls") #vector of n_outputs, each output is sensitivity matrix of n_timepoints x n_parameters
    #maximum across kb
    max_ls_kb = []
    for i in 1:n_outputs
        max_ls_kb = cat(max_ls_kb, [vec(maximum(abs.(local_sensitivity_kb[i]), dims=2))], dims=1) #take parameter with highest sensitivity for particular output
    end
    ls_kb = reshape(Base.stack(max_ls_kb), n_outputs*n_timepoints)
    serialize("outputs/600_$(baseline_run)_5_percent_change_$(i)_1d.jls", ls_kb)
end